# Tests for `dynamic_charge_controls.py`

Series of module-level tests using real inputs as much as possible for debugging and validating dynamic_charge_controls functions. 

In [ ]:
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('G:/My Drive/utils') # Unified code for several projects
from plotting import df_plot

# Import EPW library to read weather files. If not installed, this will install it. 
try:
    from epw import epw
except ModuleNotFoundError as e:
    %pip install git+https://github.com/building-energy/epw.git@master
    from epw import epw

print(f'cwd is {os.getcwd()}')
sys.path.append(os.path.join(os.getcwd(), '..', 'src', 'stor4build'))
import dynamic_charge_controls as dcc
from dynamic_charge_controls import *
dcc.debug = True # enable debug outputs for testing

In [ ]:
baseline_run_path = os.path.join(os.path.dirname(os.getcwd()), 'results', 'dynamic_charge_controls_largehotel', 'icetank', 'no_charging',  'run')
print(baseline_run_path)

epw_path = os.path.join(os.path.dirname(os.getcwd()), 'weather', 'USA_CA_Bakersfield-Meadows.Field.AP.723840_TMY3.epw')
print(epw_path)

# Check that initial file works

In [ ]:
df = read_eplusout_skip_sizing(os.path.join(os.getcwd(), '..','results', 'dynamic_charge_controls_largehotel','icetank', 'no_charging', 'run','eplusout.csv'))
# df.head()
print(len(df.columns))
for col in df.columns:
	print(col)

# Test `get_idf_info()`

In [ ]:
info = get_idf_info(os.path.join(baseline_run_path, "in.idf"))

print(info)

# Test `chiller_design_capacities()`

In [ ]:
chillercaps = get_chiller_design_capacities(os.path.join(baseline_run_path, "eplusout.eio"))

print(chillercaps)

# Test `get_icetank_specs()`
Compute ice tank specs (capacity, charge/discharge rates, loss rate) for a given number of tanks and sanity-check the resulting values.

In [ ]:
icetank_specs = get_icetank_specs(20)

for k, v in icetank_specs.items():
    print(f"{k}: {v}")

# Test `preprocess_baseline()`
Run the full baseline preprocessing step against the actual baseline run folder, using the Bakersfield epw file as a fallback for outdoor air temperature (since the baseline run's `eplusout.csv` doesn't include an OAT output variable).

This exercises `read_eplusout_skip_sizing()`, `get_idf_info()`, `get_chiller_design_capacities()`, and `get_icetank_specs()` together, and produces the hourly dataframe (`dfh`, returned as `df`) and `info` dict used by all the downstream functions tested below.

In [ ]:
df, info = preprocess_baseline(baseline_run_path, epw_file=epw_path)

print('--- info ---')
for k, v in info.items():
    print(f"{k}: {v}")

print('\n--- df ---')
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
df_plot(df.reset_index(drop=True), 'datetime', ['Thermal Load [kW]', 'Dry Bulb Temperature'], ['Electricity Rate [$/kWh]'], includelast=True, xlabel='Date', ylabel='Load [kW] / OAT [C]', y2label='Electricity Rate [$/kWh]')

# Test `generate_electricity_prices()`
Use the same default `electric_rate` / `demand_charge_schedule` / `demand_charge_rate` values that `preprocess_baseline()` falls back to internally (now stored in `info` from the previous test), and confirm the standalone function produces the same price schedule.

In [ ]:
prices = generate_electricity_prices(info['electric_rate'], info['demand_charge_schedule'], info['demand_charge_rate'], info)

print(prices.shape)
print(prices.head())
print(prices.tail())

In [ ]:
df_plot(prices, 'datetime', ['Electricity Rate [$/kWh]'], ['Demand Period'], includelast=True, plottype=['step', 'step'], xlabel='Date', ylabel='Electricity Rate [$/kWh]', y2label='Demand Period')

# Test `applyDemandCharge()`
Apply demand charges to the baseline hourly dataframe from `preprocess_baseline()` and confirm the resulting per-hour `Demand Cost`, overall `inc_cost`, and peak consumption levels (`curr_max_elec`) look reasonable (e.g. the single largest overall consumption hour should get an overall demand charge applied).

In [ ]:
df["Electricity:Facility [kW]"] = df["Electricity:Facility [W]"] / 1000
sch_test = df.loc[(df['datetime'] >= info["start_date"]) & (df['datetime'] <= info["end_date"])].reset_index(drop=True)

sch_test, curr_max_elec = applyDemandCharge(sch_test, info['demand_charge_rate'], cost='Electricity Rate [$/kWh]', elec='Electricity:Facility [kW]', demandWindow='Demand Period', demandCost='Demand Cost', debug=False)

print('curr_max_elec:', curr_max_elec)
sch_test[['datetime', 'Electricity:Facility [kW]', 'Demand Period', 'Demand Cost', 'inc_cost']].sort_values('Demand Cost', ascending=False).head(10)

# Test `generate_schedule_file()`
Call directly with a small synthetic `dms` dataframe (a handful of hours with different `mode`/`charge_temperature` values) and confirm the resulting annual, sub-hourly CSV schedule has the expected shape and correctly interpolated/forward-filled values for the specified hours.

In [ ]:
test_info = {'year': 2006, 'timesteps_per_hour': 4}
test_dms = pd.DataFrame({
    'datetime': pd.date_range('2006-01-01 00:00:00', periods=6, freq='H'),
    'mode': [0, 1, 1, -1, 0, 1],
    'charge_temperature': [6.7, -1.0, -3.8, 10, 6.7, -2.0],
})

test_output_path = os.path.join(os.getcwd(), '_test_schedule_file_output.csv')
generate_schedule_file(test_dms, test_info, test_output_path)

result = pd.read_csv(test_output_path)
print(result.shape)
print(result.head(20))
print('mode value counts:', result['mode'].value_counts().to_dict())
os.remove(test_output_path)

# Test `generate_schedule()`
Run the full end-to-end schedule generation exactly as it's called in actual usage, which internally exercises every other function above plus `consolidate_charging_hours()`. This writes `dynamic_charge_schedule.csv` to the real results folder (two levels up from `baseline_run_path`).

NOTE: This can take several minutes to run since it iterates the load-shifting optimization loop over the full 8760-hour year.

In [ ]:
output_path = generate_schedule(baseline_run_path, None, None, None, epw_path)

print('output_path:', output_path)

In [15]:
output_path = r"C:\Brian_Local\stor4build\results\dynamic_charge_controls_with_outputvars\icetank\dynamic_charge_schedule.csv"

In [ ]:
dcs = pd.read_csv(output_path)
dcs['datetime'] = pd.to_datetime(dcs['datetime'])

print(dcs.shape)
print(dcs['mode'].value_counts())
dcs.head(24)

In [ ]:
df_plot(dcs, 'datetime', ['chrg_temp'], ['mode'], includelast=True, plottype=['step', 'step'], xlabel='Date', ylabel='Charge Temperature Setpoint [C]', y2label='Mode (-1=discharge, 0=normal, 1=charge)')

Note: if anything is changed in the Python scripts, it will reflect automatically here

However, if you need to actually run the `stor4build  run-icetank-dynamic <parameters>`, first need to run this

```bash
pip uninstall stor4build -y && pip install . 
```

